In [ ]:
# Standard libraries and model imports
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_moons
from matplotlib import pyplot as plt
from MLP_classifier import MLP_classifier
from VQC_classifier import VQC_classifier
from VQC_fm_classifier import VQC_fm_classifier
from scipy.stats import ttest_rel
import os
from datetime import datetime

# Experiment Configuration
Edit **only this cell** to change what gets run.

In [ ]:
# ── Dataset ─────────────────────────────────────────────────────────────────
# make_classification generates a synthetic binary classification dataset.
# Set ACTIVE to the desired dataset config and edit 'experiment' to choose
# which parameter to sweep and over what values.

MAKE_CLASSIFICATION = dict(
    fn = make_classification,
    params = dict(
        n_samples            = 1000,
        n_features           = 4,
        n_informative        = 4,
        n_redundant          = 0,
        n_classes            = 2
    ),
    experiment = dict(
        param  = 'class_sep',   # dataset parameter to vary
        values = [0.1, 0.2, 0.3]
    )
)

ACTIVE     = MAKE_CLASSIFICATION
DATASET_FN = ACTIVE['fn']
DATASET    = ACTIVE['params']
EXPERIMENT = ACTIVE['experiment']

SEEDS = [42]  # random seeds — use multiple seeds to average out randomness

MODELS = ['mlp', 'benchmark', 'vqc']  # models to run; remove any to skip

# Comparable MLP — matches VQC parameter count for a fair comparison
MLP_CONFIG = dict(
    hidden     = (4,),
    activation = 'tanh',
    solver     = 'lbfgs',
    maxiter    = 8000
)

# Benchmark MLP — strong classical model with no parameter-count constraint
BENCHMARK_CONFIG = dict(
    hidden     = (32, 16),
    activation = 'relu',
    solver     = 'lbfgs',
    maxiter    = 8000
)

# VQC config selected from hyperparameter sweep (VQC_sweep.py)
VQC_CONFIG = dict(
    maxiter     = 100,
    fm_rep      = 1,
    ansatz_rep  = 2,
    ansatz_type = 'EfficientSU2'
)

PREPROCESS_MODE = 'scaled'  # 'raw', 'scaled', or 'pca'

N_COMPONENTS = None  # number of PCA components; only used when PREPROCESS_MODE = 'pca'

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## Run function
Trains and evaluates all selected models on a single (X, y) dataset.
Returns averaged metrics, objective value curves (VQC only), and per-fold scores.

In [ ]:
def run(X, y, seed):
    row = {}
    obj_vals = {}
    fold_scores = {}

    if 'mlp' in MODELS:
        classical = MLP_classifier(**MLP_CONFIG)
        classical.fit_predict_evaluate(X, y, preprocess_mode=PREPROCESS_MODE, n_components=N_COMPONENTS)
        fold_scores['mlp_accuracy']  = classical.get_accuracies()
        fold_scores['mlp_precision'] = classical.get_precisions()
        fold_scores['mlp_recall']    = classical.get_recalls()
        fold_scores['mlp_f1']        = classical.get_F1_Scores()
        fold_scores['mlp_time']      = classical.get_train_times()
        row.update({
            'mlp_accuracy':      np.mean(classical.get_accuracies()),
            'mlp_std_accuracy':  np.std(classical.get_accuracies()),
            'mlp_precision':     np.mean(classical.get_precisions()),
            'mlp_std_precision': np.std(classical.get_precisions()),
            'mlp_recall':        np.mean(classical.get_recalls()),
            'mlp_std_recall':    np.std(classical.get_recalls()),
            'mlp_f1':            np.mean(classical.get_F1_Scores()),
            'mlp_std_f1':        np.std(classical.get_F1_Scores()),
            'mlp_time':          np.mean(classical.get_train_times()),
        })

    if 'benchmark' in MODELS:
        benchmark = MLP_classifier(**BENCHMARK_CONFIG)
        benchmark.fit_predict_evaluate(X, y, preprocess_mode=PREPROCESS_MODE, n_components=N_COMPONENTS)
        fold_scores['benchmark_accuracy']  = benchmark.get_accuracies()
        fold_scores['benchmark_precision'] = benchmark.get_precisions()
        fold_scores['benchmark_recall']    = benchmark.get_recalls()
        fold_scores['benchmark_f1']        = benchmark.get_F1_Scores()
        fold_scores['benchmark_time']      = benchmark.get_train_times()
        row.update({
            'benchmark_accuracy':      np.mean(benchmark.get_accuracies()),
            'benchmark_std_accuracy':  np.std(benchmark.get_accuracies()),
            'benchmark_precision':     np.mean(benchmark.get_precisions()),
            'benchmark_std_precision': np.std(benchmark.get_precisions()),
            'benchmark_recall':        np.mean(benchmark.get_recalls()),
            'benchmark_std_recall':    np.std(benchmark.get_recalls()),
            'benchmark_f1':            np.mean(benchmark.get_F1_Scores()),
            'benchmark_std_f1':        np.std(benchmark.get_F1_Scores()),
            'benchmark_time':          np.mean(benchmark.get_train_times()),
        })

    if 'vqc' in MODELS:
        quantum = VQC_classifier(**VQC_CONFIG)
        quantum.fit_predict_evaluate(X, y, preprocess_mode=PREPROCESS_MODE, n_components=N_COMPONENTS)
        fold_scores['vqc_accuracy']  = quantum.get_accuracies()
        fold_scores['vqc_precision'] = quantum.get_precisions()
        fold_scores['vqc_recall']    = quantum.get_recalls()
        fold_scores['vqc_f1']        = quantum.get_F1_Scores()
        fold_scores['vqc_time']      = quantum.get_train_times()
        row.update({
            'vqc_accuracy':      np.mean(quantum.get_accuracies()),
            'vqc_std_accuracy':  np.std(quantum.get_accuracies()),
            'vqc_precision':     np.mean(quantum.get_precisions()),
            'vqc_std_precision': np.std(quantum.get_precisions()),
            'vqc_recall':        np.mean(quantum.get_recalls()),
            'vqc_std_recall':    np.std(quantum.get_recalls()),
            'vqc_f1':            np.mean(quantum.get_F1_Scores()),
            'vqc_std_f1':        np.std(quantum.get_F1_Scores()),
            'vqc_time':          np.mean(quantum.get_train_times()),
        })
        obj_vals['vqc'] = quantum.get_objective_values()

    return row, obj_vals, fold_scores

In [ ]:
for entry in all_obj_vals:                                                                                                                                     
      if 'vqc' not in entry['obj_vals']:                                                                                                                       
          continue                                                                                                                                               
      fold_curves = entry['obj_vals']['vqc']  # list of 5 fold curves
      plt.figure()                                                                                                                                               
      for i, fold in enumerate(fold_curves):                                                                                                                   
          plt.plot(fold, label=f'Fold {i+1}')                                                                                                                    
      plt.xlabel('Iteration')                                                                                                                                  
      plt.ylabel('Objective Value')                                                                                                                              
      plt.title(f"{EXPERIMENT['param']} = {entry[EXPERIMENT['param']]}, seed = {entry['seed']}")                                                                 
      plt.legend()                                                                                                                                               
      plt.grid(True, alpha=0.3)                                                                                                                                  
      plt.show()         

## Main experiment loop
Iterates over each value of the experiment parameter, regenerates the dataset
with that value, runs all models across all seeds, and saves results to CSV.
Two CSVs are saved per run: averaged results and fold-level scores.

In [ ]:
results = []
all_obj_vals = []
fold_results = []

for exp in EXPERIMENT['values']:
    print(f"\n{'='*50}")
    print(f"{EXPERIMENT['param']} = {exp}")
    print(f"{'='*50}")

    for seed in SEEDS:
        print(f"\n-- seed {seed} --")
        X, y = DATASET_FN(**DATASET, **{EXPERIMENT['param']: exp}, random_state=seed)
        row, obj_vals, fold_scores = run(X, y, seed)
        row = {EXPERIMENT['param']: exp, 'seed': seed, **row}
        results.append(row)
        all_obj_vals.append({EXPERIMENT['param']: exp, 'seed': seed, 'obj_vals': obj_vals})

        # save fold-level scores (only if fold_scores is populated)
        if fold_scores:
            k = len(list(fold_scores.values())[0])
            for fold_idx in range(k):
                fold_row = {EXPERIMENT['param']: exp, 'seed': seed, 'fold': fold_idx + 1}
                for metric, vals in fold_scores.items():
                    fold_row[metric] = vals[fold_idx]
                fold_results.append(fold_row)

raw_df = pd.DataFrame(results)
fold_df = pd.DataFrame(fold_results)
print(raw_df)

# Save CSVs first — before any groupby that might fail
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
models_str = '_'.join(MODELS)
fname      = f"{RESULTS_DIR}/{models_str}_{EXPERIMENT['param']}_{timestamp}.csv"
fname_fold = f"{RESULTS_DIR}/{models_str}_{EXPERIMENT['param']}_folds_{timestamp}.csv"
raw_df.to_csv(fname, index=False)
fold_df.to_csv(fname_fold, index=False)
print(f"Saved to {fname}")
print(f"Fold results saved to {fname_fold}")

if len(SEEDS) > 1:
    metric_cols = [c for c in raw_df.columns if c not in [EXPERIMENT['param'], 'seed']]
    group_key = raw_df[EXPERIMENT['param']].apply(lambda x: str(x) if isinstance(x, list) else x)
    final_df = raw_df.groupby(group_key)[metric_cols].agg(['mean', 'std'])
    print(final_df)
    fname_final = f"{RESULTS_DIR}/{models_str}_{EXPERIMENT['param']}_avg_{timestamp}.csv"
    final_df.to_csv(fname_final)
    print(f"Averaged results saved to {fname_final}")